# LoRA fine-tune Qwen2.5-3B-Instruct — voice assistant style

**Where to run this:** Google Colab (free T4) or Kaggle (free P100). No credit card. Runs in about 15–25 minutes on the sample dataset.

**What you'll get:**
- A LoRA adapter (~40 MB)
- Merged HF model
- Optional GGUF (quantized) for Ollama

**Before you run:** on Colab, pick `Runtime → Change runtime type → T4 GPU` first. The training cells will crash loudly if you're on CPU.

## 1. Install dependencies

In [ ]:
!pip install -q --upgrade transformers accelerate peft trl datasets bitsandbytes sentencepiece huggingface_hub

## 2. Clone the repo (or just upload the dataset)

Easiest path: clone the repo so the notebook can call `prepare_dataset.py` and `train.py` directly. If you prefer, replace this cell with an upload of just `dataset_example.jsonl`, `prepare_dataset.py`, and `train.py`.

In [ ]:
import os
REPO_URL = 'https://github.com/HemantBK/AI-Voice-Assistant.git'
if not os.path.exists('AI-Voice-Assistant'):
    !git clone $REPO_URL AI-Voice-Assistant
%cd AI-Voice-Assistant

## 3. Prepare train / eval splits

The sample dataset is 25 rows — enough to smoke-test the pipeline, not enough for a real quality gain. Grow it to 200–500 rows for meaningful results.

In [ ]:
!python finetune/prepare_dataset.py \
  --input finetune/dataset_example.jsonl \
  --out-dir finetune/data \
  --eval-frac 0.2 \
  --seed 42

## 4. Train — LoRA on Qwen2.5-3B-Instruct

Default: 3 epochs, LoRA r=16, 4-bit quantized base. On a T4 with 25 examples this is ~10 minutes.

Watch for **eval_loss going DOWN**. If it goes up after epoch 1, reduce epochs or lower learning rate. The final `metrics.json` is saved for later comparison.

In [ ]:
!python finetune/train.py \
  --train-file finetune/data/train.jsonl \
  --eval-file  finetune/data/eval.jsonl \
  --base-model Qwen/Qwen2.5-3B-Instruct \
  --out-dir    finetune/out \
  --epochs 3 \
  --batch-size 1 \
  --grad-accum 8

## 5. Quick qualitative check

The training script writes the first 3 eval-set completions to `samples.jsonl`. Read them. If the fine-tuned answer is **shorter and more on-rubric** than the reference, the style transfer worked. If it's copying the reference verbatim, you overfit — reduce epochs next time.

In [ ]:
import json, pathlib
for line in pathlib.Path('finetune/out/samples.jsonl').read_text(encoding='utf-8').splitlines():
    row = json.loads(line)
    print('Q :', row['prompt'])
    print('REF:', row['reference'])
    print('FT :', row['finetuned'])
    print('-' * 40)

## 6. Merge adapter + export for Ollama

Two options:

**(a) HF-only** — easiest. Just produces a merged model folder. Use it from Python with `AutoModelForCausalLM.from_pretrained(...)`.

**(b) GGUF + Ollama Modelfile** — best for the voice assistant. Needs llama.cpp checked out. Run the GGUF cell below only after cloning llama.cpp.

In [ ]:
# HF-only merge — always works
!python finetune/merge_export.py \
  --adapter finetune/out/adapter \
  --base-model Qwen/Qwen2.5-3B-Instruct \
  --out-dir finetune/out \
  --skip-gguf

In [ ]:
# GGUF export — needs llama.cpp. Uncomment if you want this path.
# !git clone --depth 1 https://github.com/ggerganov/llama.cpp
# !cd llama.cpp && pip install -q -r requirements.txt && cmake -B build && cmake --build build --config Release -j
# !python finetune/merge_export.py \
#   --adapter finetune/out/adapter \
#   --base-model Qwen/Qwen2.5-3B-Instruct \
#   --out-dir finetune/out \
#   --llama-cpp-dir llama.cpp \
#   --quant Q4_K_M

## 7. Download the artifacts

On Colab, the files live under `/content/AI-Voice-Assistant/finetune/out/`. The important ones:

- `adapter/` — tiny, shareable LoRA.
- `merged/` — the merged HF model (~6 GB).
- `merged.gguf` + `Modelfile` — only if you ran the GGUF cell. This is what you'll copy to the machine running Ollama.

Zip + download, or push to a HF Hub repo (`huggingface-cli login` then `huggingface-cli upload <user>/<repo> finetune/out/adapter`).

## 8. Use it locally with Ollama

On your laptop (the machine running the voice assistant):

```bash
cd finetune/out
ollama create voice-assistant-ft -f Modelfile
# test it:
ollama run voice-assistant-ft "How do I boil an egg?"
```

Then in `backend/.env`:

```
LLM_PROVIDER=ollama
OLLAMA_MODEL=voice-assistant-ft
```

Restart the backend and the voice assistant is now answering with your fine-tuned model.

## 9. A/B against the base

Run the eval harness against both models and diff the judge scores:

```bash
OLLAMA_MODEL=qwen2.5:3b           python -m eval.runners.eval_llm --judge ollama --save
OLLAMA_MODEL=voice-assistant-ft   python -m eval.runners.eval_llm --judge ollama --save
# diff the two newest eval/results/llm-*.json
```

Or use the dedicated comparison runner:

```bash
python -m eval.runners.eval_llm_compare \
  --base qwen2.5:3b \
  --finetuned voice-assistant-ft \
  --judge ollama --save
```